# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
from src.provenance import verify_checkout, verify_meridian_origin
print("Código:",verify_checkout(PROJECT_ROOT, REPO_REF))
print("Meridian:",verify_meridian_origin())
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Código: {'requested_ref': 'main', 'resolved_commit': 'ac9960a3a86e7af07a507121043c6914fcbc4195', 'dirty': True}
Meridian: {'url': 'https://github.com/google/meridian.git', 'vcs_info': {'commit_id': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'requested_revision': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'vcs': 'git'}}
Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-revision; painel: (6240, 19)


O manifesto verifica código, raw, ambiente, conclusão das etapas e hashes dos outputs; resultados incompatíveis são recalculados.

In [2]:
from src.provenance import require_stages
try:
    require_stages(range(7))
except RuntimeError as stale:
    print('Recalculando etapas anteriores:',stale)
    from src.eda import data_audit,general,temporal,relationships
    from src.geo_analysis import geo_analysis,media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit,general,temporal,geo_analysis,media_geo,relationships,official_eda][:7]:
        analysis(geo_time_panel)


## 07 — Há evidência para justificar investigar um MMM geo-level?

### Pergunta
Há evidência para justificar investigar um MMM geo-level?

### Motivação
Precisamos responder às perguntas do TCC com resultados, limites e implicações, evitando transformar descrição em causalidade.

### Método
Consolidar integridade, missing, extremos, variabilidade, população, mix, within/between, R², colinearidade e adequação oficial; produzir relatório de prontidão e narrativa final.

### Interpretação antes de olhar os resultados
Informação geo-temporal residual pode justificar a investigação hierárquica, mas não garante causalidade, generalização ou superioridade preditiva. A próxima fase requer especificação, priors, holdout e diagnósticos posteriores.

**Dependências:** painel validado e funções em `src/reporting.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.reporting import conclusions
report_07 = conclusions(geo_time_panel)
show(report_07,part="summary")

### Resultado observado e conclusão parcial

1. **Estrutura.** O painel tem 40 geos × 156 semanas = 6240 observações, completude de 100.0% e 0 células ausentes. A chave geo–tempo foi validada.

2. **Tempo.** Todos os 5 canais pagos variam; a parcela within-geo da variância fica entre 61.0% e 94.7%. Channel2 é o canal com mais zeros: 64.3% das células, com 5.5% do gasto. A tabela de atividade e os gráficos distinguem semanas sem mídia de mudanças de intensidade. Movimentos de nível e sazonalidade são exploratórios.

3. **Geografia.** As médias por geo explicam de 5.3% a 39.0% da variância bruta da mídia paga. Rankings de volume dependem de população; mix e medidas por habitante oferecem lentes complementares.

4. **GEO × TEMPO.** Após retirar médias de geo e de semana, permanece de 42.9% a 77.9% da variância bruta de mídia paga; por habitante, de 64.8% a 79.3%. Esse componente é perdido ao observar apenas o agregado nacional. O painel contém variação potencialmente informativa, mas o resíduo pode conter ruído, endogeneidade e confundimento. Comparar com a decomposição per capita é essencial: diferenças multiplicativas de tamanho também podem gerar resíduos aditivos.

5. **Identificabilidade.** R², correlações, VIF e os alertas oficiais devem orientar hipóteses de especificação. Nenhuma dessas métricas prova identificação causal ou que um modelo geo terá melhor previsão que o nacional. Essa comparação exige futura modelagem com validação temporal comum, priors justificados e diagnósticos posteriores.

**Prontidão:** a integridade permite continuar o estudo metodológico, com a revisão dos seis alertas documentada e critérios por canal/controle em readiness_actions. As limitações da simulação permanecem. Não há recomendação de alocação de orçamento nesta EDA.

**Prontidão revisada:** casos e alertas possuem decisão descritiva. Avançar ao desenho da especificação, mantendo os dados. DAG, controles, priors e validação temporal pertencem à próxima fase. Colab real continua pendente externa.

**Channel0:** zeros 12.1%; CV positivo 0.59; sequência zero 4; amplitude mix 3.30 p.p.; R² GEO bruto/per capita 0.248/0.006; resíduo per capita 0.783; VIF within 1.36; maior associação two-way sentiment_score_control (0.365). Risco: escala populacional e movimentos conjuntos não equivalem a efeito. Ação: verificar hipótese causal e colinearidade após transformações.

**Channel1:** zeros 27.9%; CV positivo 0.67; sequência zero 7; amplitude mix 3.74 p.p.; R² GEO bruto/per capita 0.144/0.004; resíduo per capita 0.765; VIF within 1.19; maior associação two-way sentiment_score_control (0.307). Risco: escala populacional e movimentos conjuntos não equivalem a efeito. Ação: verificar hipótese causal e colinearidade após transformações.

**Channel2:** zeros 64.3%; CV positivo 0.81; sequência zero 27; amplitude mix 2.42 p.p.; R² GEO bruto/per capita 0.053/0.004; resíduo per capita 0.793; VIF within 1.62; maior associação two-way Channel3_impression (0.451). Risco: suporte intermitente; janelas sem exposição. Ação: documentar suporte por janela de treino/holdout e incerteza do canal.

**Channel3:** zeros 2.7%; CV positivo 0.48; sequência zero 2; amplitude mix 3.36 p.p.; R² GEO bruto/per capita 0.390/0.005; resíduo per capita 0.648; VIF within 2.11; maior associação two-way Channel2_impression (0.451). Risco: escala populacional e movimentos conjuntos não equivalem a efeito. Ação: verificar hipótese causal e colinearidade após transformações.

**Channel4:** zeros 13.1%; CV positivo 0.60; sequência zero 4; amplitude mix 3.37 p.p.; R² GEO bruto/per capita 0.237/0.004; resíduo per capita 0.706; VIF within 1.65; maior associação two-way competitor_sales_control (0.478). Risco: escala populacional e movimentos conjuntos não equivalem a efeito. Ação: verificar hipótese causal e colinearidade após transformações.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** **Prontidão revisada:** casos e alertas possuem decisão descritiva. Avançar ao desenho da especificação, mantendo os dados. DAG, controles, priors e validação temporal pertencem à próxima fase. Colab real continua pendente externa.
- **Quais problemas apareceram?** Colab real não comprovado; EDA não demonstra identificação causal ou superioridade preditiva.
- **O que falta investigar?** Desenho causal, especificação, priors, holdout temporal e posterior em fase futura.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


### Evidências e leitura das figuras
Tabelas pequenas são exibidas integralmente. As sínteses selecionam os casos relevantes; os CSVs mantêm os dados completos.

In [4]:
show(report_07,part="evidence")

**readiness_report** — 18 linhas; CSV completo em `outputs/tables/`.

,Dimension,Question,Evidence,Finding,Modeling_implication
0,integridade,Painel completo?,data_audit,"6240 linhas, 40 geos, 156 semanas; completude 1.0",Contrato validado; preservar raw e schema.
1,missing,Há valores ausentes?,missing_summary,0 células ausentes,Não imputar dados completos.
2,outliers,Há extremos descritivos?,outliers,6039 pares observação/variável sinalizados por...,Revisão por caso e grupo em outlier_case_revie...
3,KPI variation,Existe variação?,within_between_variance,1/1 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
4,media variation,Existe variação?,within_between_variance,6/6 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
5,spend variation,Existe variação?,within_between_variance,5/5 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
6,control collinearity,Existe variação?,within_between_variance,2/2 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
7,temporal variation,Quanto varia a mídia paga?,geo_time_r2,r2_time: 13.63% a 19.07%,Avaliar também escala por população e colinear...
8,geo variation,Quanto varia a mídia paga?,geo_time_r2,r2_geo: 5.35% a 38.99%,Avaliar também escala por população e colinear...
9,within variation,Quanto varia a mídia paga?,geo_time_r2,within_share: 61.01% a 94.65%,Avaliar também escala por população e colinear...


**Fichas integradas** — 5 linhas; CSV completo em `outputs/tables/`.

,channel,zero_share,spend_share,active_share_min,active_share_max,median_positive_cv,median_unconditional_cv,max_zero_streak,max_active_streak,minimum_active_in_13_weeks,within_variance_activity_share,within_variance_intensity_share,mix_min_percent,mix_max_percent,mix_range_pp,mix_std_pp,population_corr_raw,population_corr_per_capita,r2_geo_raw,r2_geo_per_capita,residual_raw,residual_per_capita,vif_within,strongest_two_way_peer,strongest_two_way_correlation,risk,action,eda_decision
0,Channel0,0.120994,0.184524,0.782051,0.923077,0.592377,0.735606,4.0,56.0,6.0,0.253252,0.746748,16.700052,19.999401,3.299349,0.738823,0.993246,0.285741,0.248371,0.005859,0.615299,0.782822,1.361274,sentiment_score_control,0.365395,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,"manter na especificação candidata, sem exclusã..."
1,Channel1,0.279167,0.142694,0.628205,0.794872,0.673745,1.012800,7.0,25.0,3.0,0.377586,0.622414,12.359776,16.098349,3.738573,0.820257,0.987430,0.047092,0.144111,0.004375,0.687958,0.765230,1.193292,sentiment_score_control,0.307119,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,"manter na especificação candidata, sem exclusã..."
2,Channel2,0.642628,0.055033,0.282051,0.442308,0.807290,1.880868,27.0,10.0,0.0,0.504827,0.495173,4.039093,6.457172,2.418080,0.570326,0.966417,0.205816,0.053490,0.004344,0.779420,0.792821,1.616286,Channel3_impression,0.451079,suporte intermitente; janelas sem exposição,documentar suporte por janela de treino/holdou...,"manter na especificação candidata, sem exclusã..."
3,Channel3,0.027083,0.400289,0.942308,0.987179,0.484672,0.520866,2.0,127.0,9.0,0.105001,0.894999,38.001664,41.359111,3.357446,0.914462,0.993621,0.142402,0.389879,0.004927,0.429022,0.648477,2.111746,Channel2_impression,0.451079,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,"manter na especificação candidata, sem exclusã..."
4,Channel4,0.131410,0.217459,0.826923,0.935897,0.596163,0.758500,4.0,53.0,7.0,0.274841,0.725159,20.555931,23.924977,3.369046,0.790732,0.993058,0.231895,0.237213,0.004196,0.572040,0.705793,1.647581,competitor_sales_control,0.478088,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,"manter na especificação candidata, sem exclusã..."


**Ações verificáveis** — 15 linhas; CSV completo em `outputs/tables/`.

,subject,finding,risk,action,closure_criterion,status,owner,evidence
0,Channel0,zeros 12.1%; resíduo per capita 78.3%; VIF wit...,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,ficha completa e hipóteses justificadas antes ...,EDA documentada; especificação futura,autor do TCC,channel_integrated_review.csv
1,Channel1,zeros 27.9%; resíduo per capita 76.5%; VIF wit...,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,ficha completa e hipóteses justificadas antes ...,EDA documentada; especificação futura,autor do TCC,channel_integrated_review.csv
2,Channel2,zeros 64.3%; resíduo per capita 79.3%; VIF wit...,suporte intermitente; janelas sem exposição,documentar suporte por janela de treino/holdou...,ficha completa e hipóteses justificadas antes ...,EDA documentada; especificação futura,autor do TCC,channel_integrated_review.csv
3,Channel3,zeros 2.7%; resíduo per capita 64.8%; VIF with...,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,ficha completa e hipóteses justificadas antes ...,EDA documentada; especificação futura,autor do TCC,channel_integrated_review.csv
4,Channel4,zeros 13.1%; resíduo per capita 70.6%; VIF wit...,escala populacional e movimentos conjuntos não...,verificar hipótese causal e colinearidade após...,ficha completa e hipóteses justificadas antes ...,EDA documentada; especificação futura,autor do TCC,channel_integrated_review.csv
5,competitor_sales_control,R² time 0.332; VIF within 1.83,controle pós-tratamento/proxy de demanda,Justificar papel pré-tratamento/confundidor no...,DAG e temporalidade justificados,pendente da especificação,autor do TCC,control_relationship_review.csv
6,sentiment_score_control,R² time 0.274; VIF within 1.88,controle pós-tratamento/proxy de demanda,Justificar papel pré-tratamento/confundidor no...,DAG e temporalidade justificados,pendente da especificação,autor do TCC,control_relationship_review.csv
7,Promo,R² time 0.198; VIF within 1.28,controle pós-tratamento/proxy de demanda,Justificar papel pré-tratamento/confundidor no...,DAG e temporalidade justificados,pendente da especificação,autor do TCC,control_relationship_review.csv
8,REVIEW-1,36 extremos; grupos={'conversions': 36}; máxim...,"confundir cauda sintética, escala e intermitên...",preservar: limites e localizações reconciliado...,"localizações verificadas, magnitude quantifica...",encerrado para EDA simulada; REVIEW oficial pr...,autor do TCC,official_outlier_cases.csv; official_outlier_r...
9,REVIEW-2,"1461 extremos; grupos={'Channel2': 824, 'Promo...","confundir cauda sintética, escala e intermitên...",preservar: limites e localizações reconciliado...,"localizações verificadas, magnitude quantifica...",encerrado para EDA simulada; REVIEW oficial pr...,autor do TCC,official_outlier_cases.csv; official_outlier_r...


**Como ler — readiness dashboard:** Partes de variância são descritivas; contagem INFO/REVIEW não substitui decisões por alerta.

[Figura gerada; veja outputs/figures ou execute a célula.]